# Chapter 9: ConvNet architecture patterns - Colab study notebook

**Edition:** Third edition. **Execution status:** Not run in this study workspace.

[Official source](https://github.com/fchollet/deep-learning-with-python-notebooks/blob/fbf7f1bf2041093c4081e72544ec5905ace9dc8f/chapter09_convnet-architecture-patterns.ipynb) | [Official Colab copy](https://colab.research.google.com/github/fchollet/deep-learning-with-python-notebooks/blob/fbf7f1bf2041093c4081e72544ec5905ace9dc8f/chapter09_convnet-architecture-patterns.ipynb)

Upstream revision: `fbf7f1bf2041093c4081e72544ec5905ace9dc8f`. Notebook code by the official companion-notebook contributors; Copyright (c) 2017-present Francois Chollet, MIT license (see LICENSE.upstream.txt beside this notebook).

Read beside the chapter and run cells in order. Preserve the upstream backend selection: KERAS_BACKEND must be chosen before importing Keras. Restart the Colab session before changing backends. Chapter 3 demonstrates multiple frameworks; use its section-specific imports and setup.

Choose the Colab runtime for the section. Check accelerator availability before long runs. Use the upstream installation cells; no extra version pins or compatibility patches have been introduced. Use interactive login or Colab secrets when authentication is requested; do not save credentials in notebook cells.

The upstream code cells are unchanged. Saved outputs and execution counts were cleared. This study header, environment-recording cell, and run-evidence footer are the only new cells. See ../NOTEBOOK_CHANGES.md for the complete preparation record.

Preparation is not execution: only mark examples complete after running and inspecting their results.


This is a companion notebook for the book [Deep Learning with Python, Third Edition](https://www.manning.com/books/deep-learning-with-python-third-edition). For readability, it only contains runnable code blocks and section titles, and omits everything else in the book: text paragraphs, figures, and pseudocode.

**If you want to be able to follow what's going on, I recommend reading the notebook side by side with your copy of the book.**

The book's contents are available online at [deeplearningwithpython.io](https://deeplearningwithpython.io).

In [ ]:
!pip install keras keras-hub --upgrade -q

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"

In [ ]:
# Study addition: environment evidence; does not import ML frameworks.
# Run after the upstream backend-selection cell and before expensive training.
import datetime
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess

study_environment = {
    "recorded_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "python": platform.python_version(),
    "platform": platform.platform(),
    "requested_keras_backend": os.environ.get("KERAS_BACKEND", "not explicitly set"),
    "packages": {},
}
for study_package in ("keras", "tensorflow", "torch", "jax", "jaxlib", "numpy", "kagglehub", "keras-hub"):
    try:
        study_environment["packages"][study_package] = importlib.metadata.version(study_package)
    except importlib.metadata.PackageNotFoundError:
        study_environment["packages"][study_package] = "not installed"
if shutil.which("nvidia-smi"):
    study_gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, check=False,
    )
    study_environment["gpu"] = study_gpu.stdout.strip() or "GPU query unsuccessful"
else:
    study_environment["gpu"] = "nvidia-smi unavailable; check actual framework devices"
print(json.dumps(study_environment, indent=2))
# After framework imports, also record keras.backend.backend() and the framework device list.


In [ ]:
# @title
import os
from IPython.core.magic import register_cell_magic

@register_cell_magic
def backend(line, cell):
    current, required = os.environ.get("KERAS_BACKEND", ""), line.split()[-1]
    if current == required:
        get_ipython().run_cell(cell)
    else:
        print(
            f"This cell requires the {required} backend. To run it, change KERAS_BACKEND to "
            f"\"{required}\" at the top of the notebook, restart the runtime, and rerun the notebook."
        )

## ConvNet architecture patterns

### Modularity, hierarchy, and reuse

### Residual connections

In [ ]:
import keras
from keras import layers

inputs = keras.Input(shape=(32, 32, 3))
x = layers.Conv2D(32, 3, activation="relu")(inputs)
residual = x
x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
residual = layers.Conv2D(64, 1)(residual)
x = layers.add([x, residual])

In [ ]:
inputs = keras.Input(shape=(32, 32, 3))
x = layers.Conv2D(32, 3, activation="relu")(inputs)
residual = x
x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
x = layers.MaxPooling2D(2, padding="same")(x)
residual = layers.Conv2D(64, 1, strides=2)(residual)
x = layers.add([x, residual])

In [ ]:
inputs = keras.Input(shape=(32, 32, 3))
x = layers.Rescaling(1.0 / 255)(inputs)

def residual_block(x, filters, pooling=False):
    residual = x
    x = layers.Conv2D(filters, 3, activation="relu", padding="same")(x)
    x = layers.Conv2D(filters, 3, activation="relu", padding="same")(x)
    if pooling:
        x = layers.MaxPooling2D(2, padding="same")(x)
        residual = layers.Conv2D(filters, 1, strides=2)(residual)
    elif filters != residual.shape[-1]:
        residual = layers.Conv2D(filters, 1)(residual)
    x = layers.add([x, residual])
    return x

x = residual_block(x, filters=32, pooling=True)
x = residual_block(x, filters=64, pooling=True)
x = residual_block(x, filters=128, pooling=False)

x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs=inputs, outputs=outputs)

### Batch normalization

### Depthwise separable convolutions

### Putting it together: A mini Xception-like model

In [ ]:
import kagglehub

kagglehub.login()

In [ ]:
import zipfile

download_path = kagglehub.competition_download("dogs-vs-cats")

with zipfile.ZipFile(download_path + "/train.zip", "r") as zip_ref:
    zip_ref.extractall(".")

In [ ]:
import os, shutil, pathlib
from keras.utils import image_dataset_from_directory

original_dir = pathlib.Path("train")
new_base_dir = pathlib.Path("dogs_vs_cats_small")

def make_subset(subset_name, start_index, end_index):
    for category in ("cat", "dog"):
        dir = new_base_dir / subset_name / category
        os.makedirs(dir)
        fnames = [f"{category}.{i}.jpg" for i in range(start_index, end_index)]
        for fname in fnames:
            shutil.copyfile(src=original_dir / fname, dst=dir / fname)

make_subset("train", start_index=0, end_index=1000)
make_subset("validation", start_index=1000, end_index=1500)
make_subset("test", start_index=1500, end_index=2500)

batch_size = 64
image_size = (180, 180)
train_dataset = image_dataset_from_directory(
    new_base_dir / "train",
    image_size=image_size,
    batch_size=batch_size,
)
validation_dataset = image_dataset_from_directory(
    new_base_dir / "validation",
    image_size=image_size,
    batch_size=batch_size,
)
test_dataset = image_dataset_from_directory(
    new_base_dir / "test",
    image_size=image_size,
    batch_size=batch_size,
)

In [ ]:
import tensorflow as tf
from keras import layers

data_augmentation_layers = [
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.2),
]

def data_augmentation(images, targets):
    for layer in data_augmentation_layers:
        images = layer(images)
    return images, targets

augmented_train_dataset = train_dataset.map(
    data_augmentation, num_parallel_calls=8
)
augmented_train_dataset = augmented_train_dataset.prefetch(tf.data.AUTOTUNE)

In [ ]:
import keras

inputs = keras.Input(shape=(180, 180, 3))
x = layers.Rescaling(1.0 / 255)(inputs)
x = layers.Conv2D(filters=32, kernel_size=5, use_bias=False)(x)

for size in [32, 64, 128, 256, 512]:
    residual = x

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.SeparableConv2D(size, 3, padding="same", use_bias=False)(x)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.SeparableConv2D(size, 3, padding="same", use_bias=False)(x)

    x = layers.MaxPooling2D(3, strides=2, padding="same")(x)

    residual = layers.Conv2D(
        size, 1, strides=2, padding="same", use_bias=False
    )(residual)
    x = layers.add([x, residual])

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs=inputs, outputs=outputs)

In [ ]:
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)
history = model.fit(
    augmented_train_dataset,
    epochs=100,
    validation_data=validation_dataset,
)

### Beyond convolution: Vision Transformers

## Study run evidence - fill after execution

- Date and sections actually run:
- Colab runtime, accelerator, package versions (copy environment-cell output):
- Actual backend and framework devices:
- Seeds used by the original example; any seed changes and reasons:
- Input/output shapes and observed metrics:
- Errors and fixes (cell/section, original behavior, change, reason, observed result):
- Result versus the intended lesson:
- Exact resume cell/section:

Record meaningful runs in ../RUN_LOG.md and conceptual notes in ../NOTES.md. Record every environment-driven edit in ../NOTEBOOK_CHANGES.md as well. Do not claim unrun cells succeeded.
